# 🌿 LeafDoctor: MobileNetV2 Transfer Learning on Google Colab (GPU)
**Hackathon Mission 78 | AI/ML**

This notebook runs the complete training pipeline for **LeafDoctor: A Plant Doctor in Your Pocket** on a free Google Colab GPU:
1. Ingests the PlantVillage dataset across 10 target classes (Tomato, Potato, Apple).
2. Removes exact duplicates (MD5) and clusters near-duplicates to **guarantee zero data leakage** into the held-out test set.
3. Fine-tunes **MobileNetV2** in two stages (Stage 1: frozen backbone head training; Stage 2: unfreezing top layers with low lr).
4. Selects the best checkpoint based on **Validation Macro-F1**.
5. Exports `best_model.pt` ready to drop into `leafdoctor/ml/models/`.

### Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### Step 2: Install Required Libraries

In [ ]:
!pip install -q torch torchvision scikit-learn pandas pillow opencv-python matplotlib seaborn tqdm

### Step 3: Download PlantVillage Dataset
Downloads the official PlantVillage dataset color folders.

In [ ]:
import os
import shutil

# Clone PlantVillage repository directly
if not os.path.exists('PlantVillage-Dataset'):
    !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git

RAW_DATA_DIR = 'data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

# Copy target color classes into data/raw
src_color = 'PlantVillage-Dataset/raw/color'
for folder in os.listdir(src_color):
    if any(crop in folder for crop in ['Tomato', 'Potato', 'Apple']):
        dst = os.path.join(RAW_DATA_DIR, folder)
        if not os.path.exists(dst):
            shutil.copytree(os.path.join(src_color, folder), dst)
            print(f"Copied: {folder}")

print(f"\nClasses ready in {RAW_DATA_DIR}: {len(os.listdir(RAW_DATA_DIR))}")

### Step 4: Class Definitions & Configuration

In [ ]:
SUPPORTED_CLASSES = [
    {"id": "tomato_healthy", "folder_name": "Tomato___healthy", "crop": "Tomato", "condition": "Healthy", "label": "Tomato - Healthy", "is_healthy": True},
    {"id": "tomato_early_blight", "folder_name": "Tomato___Early_blight", "crop": "Tomato", "condition": "Early Blight", "label": "Tomato - Early Blight", "is_healthy": False},
    {"id": "tomato_late_blight", "folder_name": "Tomato___Late_blight", "crop": "Tomato", "condition": "Late Blight", "label": "Tomato - Late Blight", "is_healthy": False},
    {"id": "tomato_leaf_mold", "folder_name": "Tomato___Leaf_Mold", "crop": "Tomato", "condition": "Leaf Mold", "label": "Tomato - Leaf Mold", "is_healthy": False},
    {"id": "potato_healthy", "folder_name": "Potato___healthy", "crop": "Potato", "condition": "Healthy", "label": "Potato - Healthy", "is_healthy": True},
    {"id": "potato_early_blight", "folder_name": "Potato___Early_blight", "crop": "Potato", "condition": "Early Blight", "label": "Potato - Early Blight", "is_healthy": False},
    {"id": "potato_late_blight", "folder_name": "Potato___Late_blight", "crop": "Potato", "condition": "Late Blight", "label": "Potato - Late Blight", "is_healthy": False},
    {"id": "apple_healthy", "folder_name": "Apple___healthy", "crop": "Apple", "condition": "Healthy", "label": "Apple - Healthy", "is_healthy": True},
    {"id": "apple_scab", "folder_name": "Apple___Apple_scab", "crop": "Apple", "condition": "Apple Scab", "label": "Apple - Apple Scab", "is_healthy": False},
    {"id": "apple_black_rot", "folder_name": "Apple___Black_rot", "crop": "Apple", "condition": "Black Rot", "label": "Apple - Black Rot", "is_healthy": False},
]

RANDOM_SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

### Step 5: Run Deduplication & 70/15/15 Anti-Leakage Split

In [ ]:
import hashlib
import random
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

def compute_md5(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            hasher.update(chunk)
    return hasher.hexdigest()

def compute_dhash(image_path, hash_size=8):
    try:
        with Image.open(image_path) as img:
            resized = img.convert('L').resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
            pixels = np.asarray(resized)
            diff = pixels[:, 1:] > pixels[:, :-1]
            decimal_value = 0
            for index, value in enumerate(diff.flatten()):
                if value:
                    decimal_value += 1 << index
            return decimal_value
    except Exception:
        return 0

def hamming_distance(h1, h2):
    x = (h1 ^ h2) & ((1 << 64) - 1)
    tot = 0
    while x:
        tot += 1
        x &= x - 1
    return tot

# Perform deduplication and splitting
all_train, all_val, all_test = [], [], []
os.makedirs('data/processed', exist_ok=True)

print("Deduplicating and splitting dataset...")
for idx, cls in enumerate(SUPPORTED_CLASSES):
    cls_dir = os.path.join(RAW_DATA_DIR, cls['folder_name'])
    if not os.path.exists(cls_dir):
        continue
    
    files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    seen_md5 = set()
    unique_items = []
    for fp in files:
        h = compute_md5(fp)
        if h not in seen_md5:
            seen_md5.add(h)
            unique_items.append({'filepath': fp, 'class_id': idx, 'label': cls['label'], 'dhash': compute_dhash(fp)})
            
    # Cluster near duplicates
    assigned = set()
    clusters = []
    for i in range(len(unique_items)):
        if i in assigned: continue
        cluster = [i]
        assigned.add(i)
        for j in range(i + 1, len(unique_items)):
            if j not in assigned and hamming_distance(unique_items[i]['dhash'], unique_items[j]['dhash']) <= 3:
                cluster.append(j)
                assigned.add(j)
        clusters.append(cluster)
        
    # Split clusters
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(clusters)
    target_train = int(len(unique_items) * 0.70)
    target_val = int(len(unique_items) * 0.15)
    
    t_items, v_items, te_items = [], [], []
    for cl in clusters:
        items = [unique_items[i] for i in cl]
        if len(t_items) + len(items) <= target_train:
            t_items.extend(items)
        elif len(v_items) + len(items) <= target_val:
            v_items.extend(items)
        else:
            te_items.extend(items)
            
    all_train.extend(t_items)
    all_val.extend(v_items)
    all_test.extend(te_items)
    print(f"{cls['label']:<24}: Unique: {len(unique_items)} -> Train: {len(t_items)}, Val: {len(v_items)}, Test: {len(te_items)}")

pd.DataFrame(all_train).to_csv('data/processed/train.csv', index=False)
pd.DataFrame(all_val).to_csv('data/processed/val.csv', index=False)
pd.DataFrame(all_test).to_csv('data/processed/test.csv', index=False)
print(f"\nSaved: Train={len(all_train)}, Val={len(all_val)}, Test={len(all_test)}")

### Step 6: Dataset & Augmentations Setup

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from sklearn.metrics import f1_score, accuracy_score

class LeafDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['filepath']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['class_id'])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.15, 0.15, 0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(LeafDataset(pd.DataFrame(all_train), train_tf), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(LeafDataset(pd.DataFrame(all_val), val_tf), batch_size=32, shuffle=False, num_workers=2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"DataLoaders ready on {device.upper()}")

### Step 7: Two-Stage Fine-Tuning (MobileNetV2)
- **Stage 1 (5 epochs):** Freeze backbone, train new classification head ($lr = 10^{-3}$).
- **Stage 2 (10 epochs):** Unfreeze top layers, train with small learning rate ($lr = 10^{-5}$) with CosineAnnealing.

In [ ]:
num_classes = len(SUPPORTED_CLASSES)
model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)
criterion = nn.CrossEntropyLoss()

# STAGE 1
print("=== STAGE 1: Training Classification Head ===")
for p in model.features.parameters():
    p.requires_grad = False

opt1 = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
best_val_f1 = 0.0

for epoch in range(1, 6):
    model.train()
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        opt1.zero_grad()
        out = model(imgs)
        loss = criterion(out, targets)
        loss.backward()
        opt1.step()
        
    # Validate
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            trues.extend(targets.numpy())
            
    f1 = f1_score(trues, preds, average='macro', zero_division=0)
    acc = accuracy_score(trues, preds)
    print(f"Stage 1 Epoch {epoch}/5 | Val Acc: {acc*100:.1f}% | Val Macro-F1: {f1:.4f}")
    if f1 > best_val_f1:
        best_val_f1 = f1
        torch.save(model, 'best_model.pt')

# STAGE 2
print("\n=== STAGE 2: Fine-Tuning Top Blocks ===")
for i, child in enumerate(model.features.children()):
    if i >= 14:
        for p in child.parameters():
            p.requires_grad = True

opt2 = torch.optim.Adam([
    {'params': model.features[14:].parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 5e-5},
])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=10)

for epoch in range(1, 11):
    model.train()
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        opt2.zero_grad()
        out = model(imgs)
        loss = criterion(out, targets)
        loss.backward()
        opt2.step()
    sched.step()
    
    # Validate
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            trues.extend(targets.numpy())
            
    f1 = f1_score(trues, preds, average='macro', zero_division=0)
    acc = accuracy_score(trues, preds)
    print(f"Stage 2 Epoch {epoch}/10 | Val Acc: {acc*100:.1f}% | Val Macro-F1: {f1:.4f}")
    if f1 > best_val_f1:
        best_val_f1 = f1
        torch.save(model, 'best_model.pt')

print(f"\nBest Validation Macro-F1: {best_val_f1:.4f}")
print("Model saved to best_model.pt")

### Step 8: Download Checkpoint to Local Machine
Download `best_model.pt` and place it in your local repository inside `leafdoctor/ml/models/best_model.pt`.

In [ ]:
from google.colab import files
files.download('best_model.pt')